# ABO Adaptive RAG · Qwen3-VL-8B 公开体验

1. 登录 Kaggle，选择 **Copy & Edit** 建立自己的副本。
2. 在 **Session options** 启用 **Internet** 和 NVIDIA GPU（可选 T4 ×2 或可用的 L4），保留附加的 Qwen3-VL 8B Instruct 模型。
3. 运行下面的初始化单元，等到「ABO 商品助手已就绪」。首次启动会下载公开数据并建立 147,702 条商品索引。
4. 在本机启动 [ABO 网页](https://github.com/AaronYue77/ABO-Adaptive-RAG#打开网页问答台)。保持 Kaggle 会话运行，在 **Run → Kaggle Jupyter Server** 复制自己的 **VSCode Compatible URL**，粘贴到网页的「连接 Kaggle GPU」。这个地址包含当前会话凭据。

也可以在本 Notebook 新建单元直接用 `ask("给 iPhone 8 Plus 找个手机壳")`。问答日志保存在当前会话中。下面的验收单元只使用固定公开演示题。

[ABO 数据](https://amazon-berkeley-objects.s3.amazonaws.com/index.html) · [Qwen 底座](https://www.kaggle.com/models/qwen-lm/qwen-3-vl/transformers/8b-instruct/1) · [正式 LoRA](https://github.com/AaronYue77/ABO-Adaptive-RAG/releases/tag/qwen3-vl-8b-abo-lora-v1)


In [ ]:
import runpy
import sys
from pathlib import Path
from shutil import copyfileobj
from tempfile import TemporaryDirectory
from urllib.request import urlopen
from zipfile import ZipFile

revision = "8967d07887d2801fbce29d1bbafa7d5def4b0d24"
project = Path("/tmp/abo-adaptive-public") / revision
if not project.is_dir():
    project.parent.mkdir(parents=True, exist_ok=True)
    with TemporaryDirectory(dir=project.parent) as staging:
        staging = Path(staging)
        bundle = staging / "source.zip"
        url = f"https://codeload.github.com/AaronYue77/ABO-Adaptive-RAG/zip/{revision}"
        with urlopen(url, timeout=120) as response, bundle.open("wb") as stream:
            copyfileobj(response, stream)
        with ZipFile(bundle) as source:
            source.extractall(staging)
        extracted, = [path for path in staging.iterdir() if path.is_dir()]
        extracted.rename(project)

if str(project) not in sys.path:
    sys.path.insert(0, str(project))
if "chat" not in globals() or chat.journal.closed:
    bootstrap = runpy.run_path(str(project / "scripts/kaggle_abo_public.py"))
    chat = bootstrap["start"]()

def ask(question, *, details=False):
    row = chat.respond(question)
    print(row["display_answer"])
    for label in dict.fromkeys(item["label"] for item in row["display_sources"]):
        print("来源", label)
    if details:
        return row


In [ ]:
_ = bootstrap["verify"](chat)
